[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C62_Coding_Interview_Course/03_trees_graphs/03_trees_graphs.ipynb)

# 03 · 树、图与搜索（递归转迭代 / 并查集 / 连通域标记 / NMS 即极大独立集）

目标：把「树图题」从背模板变成三件可验证的能力 ——
**能把任何递归改写成显式栈**、**能认出网格题就是连通域标记**、
**能用图论语言解释 NMS 为什么是贪心近似**。

本 notebook 你会亲手实现：

1. **递归三要素的三个坑**：返回值 vs 答案（树直径）、$O(n^2)$ 的判平衡、**栈深度实测爆栈**
2. **四种遍历的递归与迭代双实现**，在 2000 棵随机树上**逐一对拍**；
   后序两种写法（前序+反转 / **stage 状态机**）；用状态机迭代做树形 DP
3. **BST 验证的两种正解 + 一种经典错解**（只比孩子），用反例把错解抓出来
4. **BFS 最短路 / 多源 BFS（= 距离变换）/ Dijkstra 惰性删除 / 0-1 BFS**，
   与 Floyd 暴力全源最短路对拍
5. **网格连通域标记两条路线对拍**：BFS 泛洪一遍法 vs **逐像素两遍扫描 + 并查集**；
   4/8 邻域给出不同答案的最小反例；mask → bbox
6. **并查集**（路径压缩 + 按秩合并）：把「不优化会退化成链」跑成数字（步数差 1000 倍）；
   **用于 bbox 聚类**并与逐对比较暴力法对拍；**演示 chaining 失效模式**
7. **拓扑排序 Kahn 分层 + 三色环检测**，与全排列暴力对拍；DAG 关键路径
8. **把 NMS 表述成图上的极大独立集**：暴力枚举求最大权独立集（MWIS），
   量化贪心的次优程度，并验证 $\mathrm{greedy} \ge \mathrm{OPT}/(\Delta+1)$

> 心智模型：**递归 = 隐式栈；把栈显式化，控制流就变成了可检查、可持久化、可限长的数据。
> 而图算法在 CV 里几乎总是换了个名字：flood fill、连通域、框聚类、冲突图。**

> 与 **C61 模块 05** 的分工：IoU / NMS / mAP 的手撕实现在那里，本 notebook 只用它们做**结构分析**。

## 1 · 递归三要素与它的三个隐性代价

先建一棵树，然后把三个最常见的坑各跑一遍：
**返回值与答案混淆**、**重算导致的 $O(n^2)$**、**栈深度**。

In [ ]:
import sys, time, random, heapq, itertools
from collections import deque, defaultdict, Counter
import numpy as np

print('Python', sys.version.split()[0], '| numpy', np.__version__,
      '| 递归上限', sys.getrecursionlimit())


class Node:
    __slots__ = ('val', 'left', 'right')
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right


def build(vals):
    """从层序列表建树，None 表示空位（LeetCode 的标准输入格式）。"""
    if not vals or vals[0] is None:
        return None
    root = Node(vals[0]); q = deque([root]); i = 1
    while q and i < len(vals):
        node = q.popleft()
        for side in ('left', 'right'):
            if i < len(vals):
                v = vals[i]; i += 1
                if v is not None:
                    child = Node(v); setattr(node, side, child); q.append(child)
    return root


def rand_tree(rnd, n):
    """随机形状二叉树，结点值是 0..n-1 的一个排列（值互不相同 ⇒ 遍历序列可直接比较）。"""
    if n == 0:
        return None
    vals = list(range(n)); rnd.shuffle(vals)
    root = Node(vals[0]); nodes = [root]
    for v in vals[1:]:
        while True:                        # n 个结点总有 n+1 个空位，必然能插进去
            p = rnd.choice(nodes)
            if rnd.random() < 0.5:
                if p.left is None:
                    p.left = Node(v); nodes.append(p.left); break
            else:
                if p.right is None:
                    p.right = Node(v); nodes.append(p.right); break
    return root


def unlink_chain(node):
    """迭代解链：超长链不能靠引用计数级联释放 —— 那是 C 栈上的递归析构，会段错误。
       这也是「后序遍历用于资源释放」的现实版本。"""
    while node is not None:
        nxt, node.left = node.left, None
        node = nxt


def nodes_of(root):
    """层序收集全部结点（迭代，安全）。"""
    out, q = [], deque([root] if root else [])
    while q:
        n = q.popleft(); out.append(n)
        if n.left:  q.append(n.left)
        if n.right: q.append(n.right)
    return out


T = build([1, 2, 3, 4, 5, None, 6])
print('测试树 =', [n.val for n in nodes_of(T)], '（层序）')
assert len(nodes_of(T)) == 6
print('✅ 树的构造与层序收集就绪。')

In [ ]:
# ── 坑 ①：返回值 ≠ 答案（树直径）──

def diameter(root):
    """直径 = 任意两结点间最长路径的**边数**。
       契约：内层 depth() 返回「以 node 为根向下的最大深度」；
             **答案挂在外部变量 best 上** —— 这两个量不是一回事。"""
    best = 0
    def depth(node):
        nonlocal best
        if node is None:
            return 0                       # 终止条件
        l, r = depth(node.left), depth(node.right)
        best = max(best, l + r)            # 经过 node 的最长路径（合并）
        return 1 + max(l, r)               # 返回值：深度，绝不是 l + r
    depth(root)
    return best


def diameter_brute(root):
    """暴力真值：把树当无向图，从每个结点跑一次 BFS，取最大距离。"""
    ns = nodes_of(root)
    if not ns:
        return 0
    adj = defaultdict(list)
    for n in ns:
        for c in (n.left, n.right):
            if c:
                adj[id(n)].append(id(c)); adj[id(c)].append(id(n))
    best = 0
    for s in ns:
        dist = {id(s): 0}; q = deque([id(s)])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if v not in dist:
                    dist[v] = dist[u] + 1; q.append(v)
        best = max(best, max(dist.values()))
    return best


# ── 坑 ②：判平衡写成「每层重算高度」→ O(n^2) ──
H_CALLS = 0
def height(node):
    global H_CALLS
    H_CALLS += 1
    return 0 if node is None else 1 + max(height(node.left), height(node.right))

def balanced_slow(node):
    if node is None:
        return True
    return (abs(height(node.left) - height(node.right)) <= 1
            and balanced_slow(node.left) and balanced_slow(node.right))

def balanced_fast(node):
    """后序一趟：同时返回 (高度, 是否平衡)。这就是「自下而上聚合」的标准形状。"""
    def go(n):
        if n is None:
            return 0, True
        hl, bl = go(n.left)
        hr, br = go(n.right)
        return 1 + max(hl, hr), (bl and br and abs(hl - hr) <= 1)
    return go(node)[1]


def sum_heights_slow(node):
    """❌ 「在每个结点上重新算一遍高度」的典型写法 —— 重复计算，最坏 O(n^2)。"""
    if node is None:
        return 0
    return height(node) + sum_heights_slow(node.left) + sum_heights_slow(node.right)

def sum_heights_fast(node):
    """✅ 后序一趟：高度在返回值里往上带，顺手累加。O(n)。"""
    total = 0
    def go(n):
        nonlocal total
        if n is None:
            return 0
        h = 1 + max(go(n.left), go(n.right))
        total += h
        return h
    go(node)
    return total


rnd_t = random.Random(0)
for _ in range(500):
    t = rand_tree(rnd_t, rnd_t.randint(0, 12))
    assert diameter(t) == diameter_brute(t)
    assert balanced_slow(t) == balanced_fast(t)
    assert sum_heights_slow(t) == sum_heights_fast(t)
print('✅ 500 棵随机树：直径与暴力 BFS 一致；两种判平衡、两种高度和结果一致。')

# 把「重复计算」跑成数字：左偏斜树上，slow 版的 height 调用次数是二次的
print('\n%5s %18s %14s' % ('n', 'slow 的 height 调用', '≈ n^2'))
for n in (50, 100, 200, 400):
    chain = None
    for v in range(n):
        chain = Node(v, left=chain)        # 纯左链 —— 重复计算的最坏形状
    H_CALLS = 0; sum_heights_slow(chain); slow_calls = H_CALLS
    H_CALLS = 0; sum_heights_fast(chain); fast_calls = H_CALLS
    print('%5d %18d %14d   （fast 版的 height 调用 = %d）' % (n, slow_calls, n * n, fast_calls))
    assert slow_calls > 0.4 * n * n, 'slow 版确实是 O(n^2)'
    assert fast_calls == 0, 'fast 版根本不调用 height —— 高度在返回值里往上带'
    unlink_chain(chain)
print('\n✅ 结论：「在每个结点上重新算高度」= O(n^2)，「后序一趟把高度带上来」= O(n)。')
print('   判平衡是同一个坑的小一号版本（最坏 O(n log n)）：')
print('   面试话术是「不能每层重算高度，要后序一趟同时返回高度和平衡标志」。')

In [ ]:
# ── 坑 ③：栈深度 —— 把 RecursionError 跑出来 ──

def depth_rec(node):
    if node is None:
        return 0
    return 1 + max(depth_rec(node.left), depth_rec(node.right))

def depth_iter(root):
    """显式栈版：深度多大都不会爆。栈里存 (结点, 该结点的深度)。"""
    if root is None:
        return 0
    best, st = 0, [(root, 1)]
    while st:
        node, d = st.pop()
        best = max(best, d)
        if node.left:  st.append((node.left, d + 1))
        if node.right: st.append((node.right, d + 1))
    return best


N_DEEP = 3 * sys.getrecursionlimit()        # 稳超上限，不依赖具体环境
chain = None
for v in range(N_DEEP):
    chain = Node(v, left=chain)

blew_up = False
try:
    depth_rec(chain)
except RecursionError:
    blew_up = True
print('深度 %d 的左链：' % N_DEEP)
print('  递归版 →', 'RecursionError（默认上限 %d）' % sys.getrecursionlimit() if blew_up else '居然没炸')
assert blew_up, '递归版必须在这里爆栈 —— 这就是本节要证明的事'
print('  迭代版 →', depth_iter(chain))
assert depth_iter(chain) == N_DEEP

t0 = time.perf_counter(); depth_iter(chain); t_it = time.perf_counter() - t0
print('  迭代版耗时 %.1f ms' % (t_it * 1000))

unlink_chain(chain); chain = None           # 迭代解链释放（见上一格的 unlink_chain）

print('\n❗ 反面教材：sys.setrecursionlimit(10**6) 不是解法。')
print('   Python 的上限是为了保护真实的 C 栈（通常 8 MB）；调过头之后爆的是 C 栈，')
print('   表现为**段错误 / 进程被杀，没有异常、没有回溯、日志里什么都没有**。')
print('   车端的感知进程崩了就是输出断流 —— 所以深度不可控时一律写迭代版。')

## 2 · 递归转迭代：前序 / 中序 / 后序 / 树形 DP

四段代码，一个共同结构：**显式栈就是把「同一个结点被经过三次」这件事写出来**。

- 前序：进入时做事 → 结点弹出后不再需要 → **单栈够用**（先压右后压左）
- 中序：左子树完成时做事 → 需要「一路向左把祖先链压栈」
- 后序：两个孩子都完成才做事 → **必须区分「第一次见」和「孩子回来了」** → 带 `stage`
- 树形 DP：后序 + 携带子树返回值 → 只有 `stage` 版本能做（「前序+反转」的技巧在这里失效）

In [ ]:
# ── 递归版（真值）──
def preorder_rec(n):  return [] if n is None else [n.val] + preorder_rec(n.left) + preorder_rec(n.right)
def inorder_rec(n):   return [] if n is None else inorder_rec(n.left) + [n.val] + inorder_rec(n.right)
def postorder_rec(n): return [] if n is None else postorder_rec(n.left) + postorder_rec(n.right) + [n.val]


# ── 迭代版 ──
def preorder_iter(root):
    out, st = [], ([root] if root else [])
    while st:
        n = st.pop(); out.append(n.val)
        if n.right: st.append(n.right)     # 先压右、后压左 ⇒ 栈是 LIFO，弹出顺序才是 左→右
        if n.left:  st.append(n.left)
    return out


def inorder_iter(root):
    out, st, cur = [], [], root
    while st or cur:
        while cur:                         # 一路向左：把整条祖先链压栈
            st.append(cur); cur = cur.left
        n = st.pop(); out.append(n.val)    # 弹出 ⇔ 「左子树已完成」这一时刻
        cur = n.right
    return out


def postorder_iter_rev(root):
    """写法 1：按 根→右→左 做前序，最后整体反转。最短、好背。
       局限：它只产出**访问序列**，无法在后序时刻使用子树的返回值。"""
    out, st = [], ([root] if root else [])
    while st:
        n = st.pop(); out.append(n.val)
        if n.left:  st.append(n.left)      # 注意左右顺序与前序相反
        if n.right: st.append(n.right)
    return out[::-1]


def postorder_iter_stage(root):
    """写法 2：栈里存 (结点, stage)。stage 就是递归的「返回点编号」：
       0 = 还没展开，1 = 左子树回来了，2 = 右子树也回来了。通用、可携带返回值。"""
    out, st = [], ([(root, 0)] if root else [])
    while st:
        n, stage = st.pop()
        if stage == 0:
            st.append((n, 1))
            if n.left:  st.append((n.left, 0))
        elif stage == 1:
            st.append((n, 2))
            if n.right: st.append((n.right, 0))
        else:
            out.append(n.val)              # 两个孩子都完成 ⇒ 后序时刻
    return out


rnd_i = random.Random(1)
for _ in range(2000):
    t = rand_tree(rnd_i, rnd_i.randint(0, 14))
    assert preorder_iter(t)       == preorder_rec(t)
    assert inorder_iter(t)        == inorder_rec(t)
    assert postorder_iter_rev(t)  == postorder_rec(t)
    assert postorder_iter_stage(t) == postorder_rec(t)
print('✅ 2000 棵随机树：四个迭代版与递归版**逐序列完全一致**。')
print('   T =', [n.val for n in nodes_of(T)], '（层序）')
print('   前序', preorder_iter(T), ' 中序', inorder_iter(T), ' 后序', postorder_iter_stage(T))

In [ ]:
# ── stage 状态机做树形 DP：迭代地算出每个结点的子树和 ──

def subtree_sums_rec(root):
    sums = {}
    def go(n):
        if n is None:
            return 0
        s = n.val + go(n.left) + go(n.right)
        sums[id(n)] = s
        return s
    go(root)
    return sums


def subtree_sums_iter(root):
    """和 postorder_iter_stage 同一个骨架，只是在 stage==2 时读取孩子的结果。
       这就是「任何递归都能机械地转成显式栈」的完整示范。"""
    sums = {}
    st = [(root, 0)] if root else []
    while st:
        n, stage = st.pop()
        if stage == 0:
            st.append((n, 1))
            if n.left:  st.append((n.left, 0))
        elif stage == 1:
            st.append((n, 2))
            if n.right: st.append((n.right, 0))
        else:
            sums[id(n)] = n.val + sums.get(id(n.left), 0) + sums.get(id(n.right), 0)
    return sums


rnd_d = random.Random(2)
for _ in range(1000):
    t = rand_tree(rnd_d, rnd_d.randint(0, 14))
    a, b = subtree_sums_rec(t), subtree_sums_iter(t)
    assert a == b, '两种实现的子树和字典必须逐键相等'
    if t is not None:
        assert a[id(t)] == sum(n.val for n in nodes_of(t))   # 根的子树和 = 全部结点之和
print('✅ 1000 棵随机树：迭代树形 DP 与递归结果逐键一致。')

# 深链上迭代版照样работает，递归版爆栈
N2 = 2 * sys.getrecursionlimit()
chain2 = None
for v in range(1, N2 + 1):
    chain2 = Node(v, left=chain2)
try:
    subtree_sums_rec(chain2); rec_ok = True
except RecursionError:
    rec_ok = False
it = subtree_sums_iter(chain2)
print('深度 %d 的链：递归版成功？%s   迭代版根的子树和 = %d（应为 1+2+...+%d = %d）'
      % (N2, rec_ok, it[id(chain2)], N2, N2 * (N2 + 1) // 2))
assert not rec_ok and it[id(chain2)] == N2 * (N2 + 1) // 2
unlink_chain(chain2); chain2 = None
print('\n📌 转换配方（背这三行就够）：栈存 (状态, 阶段)；阶段 0 展开、阶段 k 处理第 k 个子结果；')
print('   子结果放在一个以「结点身份」为键的表里。任何递归都能这样机械地转过来 ——')
print('   包括模块 04 的记忆化搜索。')

## 3 · 层序遍历与 BST：两个必考性质

层序的唯一要点：**进入循环先记下 `n = len(queue)`，只处理这 `n` 个**（它们恰好是当前整层）。
不记 `n` 就没有「层」的概念，「每层最大值」「之字形」「右视图」全都写不出来。

BST 的唯一要点：**中序遍历严格递增**。「只比较结点与它的两个孩子」是经典错解。

In [ ]:
def level_order(root):
    out, q = [], deque([root] if root else [])
    while q:
        n_this = len(q)                    # ← 关键的一行：当前层的宽度
        level = []
        for _ in range(n_this):
            node = q.popleft(); level.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        out.append(level)
    return out


def zigzag(root):
    return [lv if i % 2 == 0 else lv[::-1] for i, lv in enumerate(level_order(root))]

def right_view(root):
    return [lv[-1] for lv in level_order(root)]


L = level_order(T)
print('层序   =', L)
print('之字形 =', zigzag(T))
print('右视图 =', right_view(T))
print('每层最大值 =', [max(lv) for lv in L])
assert L == [[1], [2, 3], [4, 5, 6]]
assert zigzag(T) == [[1], [3, 2], [4, 5, 6]]
assert right_view(T) == [1, 3, 6]

rnd_l = random.Random(3)
for _ in range(1000):
    t = rand_tree(rnd_l, rnd_l.randint(0, 14))
    lv = level_order(t)
    assert sum(len(x) for x in lv) == len(nodes_of(t))       # 每个结点恰好出现一次
    assert len(lv) == depth_iter(t)                          # 层数 == 树高
    assert sorted(v for x in lv for v in x) == sorted(n.val for n in nodes_of(t))
print('✅ 1000 棵随机树：层数 == 树高，且每个结点恰好被分到一层。')
print('   空间复杂度对比：BFS 是 O(最大层宽)，DFS 是 O(树高) ——')
print('   深而窄的树用 DFS，浅而宽的树用 BFS。这是真实的工程判据。')

In [ ]:
# ── BST 验证：两种正解 + 一种经典错解 ──

def is_bst_inorder(root):
    """解法 A（中序）：只需记住「上一个访问的值」，必须严格递增。"""
    prev = None
    for v in inorder_iter(root):
        if prev is not None and v <= prev:
            return False
        prev = v
    return True


def is_bst_bounds(node, lo=None, hi=None):
    """解法 B（前序）：把 (lo, hi) 开区间约束自上而下传给孩子。"""
    if node is None:
        return True
    if lo is not None and node.val <= lo:
        return False
    if hi is not None and node.val >= hi:
        return False
    return (is_bst_bounds(node.left, lo, node.val)
            and is_bst_bounds(node.right, node.val, hi))


def is_bst_naive(node):
    """❌ 经典错解：只比较结点与它的两个孩子。局部合法 ≠ 全局合法。"""
    if node is None:
        return True
    if node.left and node.left.val >= node.val:
        return False
    if node.right and node.right.val <= node.val:
        return False
    return is_bst_naive(node.left) and is_bst_naive(node.right)


BAD = build([10, 5, 15, None, 12])       # 12 在根的**左**子树里，却比根大
print('反例树：根 10，左孩子 5，5 的右孩子 12')
print('  中序序列 =', inorder_iter(BAD), ' → 不是递增')
print('  is_bst_inorder =', is_bst_inorder(BAD), ' is_bst_bounds =', is_bst_bounds(BAD),
      ' is_bst_naive =', is_bst_naive(BAD), ' ← 错解说它是 BST')
assert is_bst_inorder(BAD) is False and is_bst_bounds(BAD) is False
assert is_bst_naive(BAD) is True, '这就是错解的失效点'


def bst_insert(root, v):
    if root is None:
        return Node(v)
    if v < root.val:
        root.left = bst_insert(root.left, v)
    elif v > root.val:
        root.right = bst_insert(root.right, v)
    return root


rnd_b = random.Random(4)
wrong = 0
for _ in range(1500):
    if rnd_b.random() < 0.5:                                  # 一半是真 BST
        t = None
        for v in rnd_b.sample(range(30), rnd_b.randint(0, 10)):
            t = bst_insert(t, v)
    else:                                                     # 一半是随机树
        t = rand_tree(rnd_b, rnd_b.randint(0, 10))
    a, b, c = is_bst_inorder(t), is_bst_bounds(t), is_bst_naive(t)
    assert a == b, '两种正解必须一致'
    if c != a:
        wrong += 1
print('\n✅ 1500 棵树：两种正解结果完全一致；错解有 %d 棵判错。' % wrong)
assert wrong > 0
print('   面试加分说法：「有两种写法 —— 中序必须严格递增，或者前序向下传 (lo, hi) 约束；')
print('   只比孩子是错的，反例是「左子树里藏一个比根大的结点」。」')

## 4 · BFS / 多源 BFS / Dijkstra / 0-1 BFS

四段代码，一条主线：**BFS 的第一次到达即最优（边权相同）；边权不同就必须换出队规则。**

- BFS：队列，**入队时标记 visited**（不是出队时）
- 多源 BFS：所有源点一次性入队 → 得到「到最近源点的距离」= **距离变换**
- Dijkstra：堆 + **惰性删除**（不 decrease-key，出堆时用 `d > dist[u]` 过滤过期条目）
- 0-1 BFS：权 0 走 `appendleft`、权 1 走 `append`，双端队列足以维持单调性 → 线性

In [ ]:
def bfs_dist(adj, s):
    """无权图单源最短路。dist 同时充当 visited —— 一个字典办两件事。"""
    dist = {s: 0}; q = deque([s])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in dist:              # ★ 入队时就标记；写在出队时会导致重复入队
                dist[v] = dist[u] + 1
                q.append(v)
    return dist


def bfs_dist_buggy(adj, s):
    """❌ 出队时才标记：同一个结点被反复入队，队列规模从 O(V) 涨到 O(E)。"""
    dist = {}; q = deque([(s, 0)]); pushes = 0
    while q:
        u, d = q.popleft()
        if u in dist:
            continue
        dist[u] = d
        for v in adj[u]:
            q.append((v, d + 1)); pushes += 1
    return dist, pushes


def dijkstra(adjw, s):
    """边权非负的单源最短路。O((V+E) log V)。"""
    INF = float('inf')
    dist = {s: 0.0}; heap = [(0.0, s)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist.get(u, INF):
            continue                       # ★ 惰性删除：过期条目直接跳过（代替 decrease-key）
        for v, w in adjw[u]:
            nd = d + w
            if nd < dist.get(v, INF):
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return dist


def bfs01(adjw, s):
    """边权只有 0/1 时的线性算法：双端队列维持「队内距离至多两个相邻取值」。"""
    INF = float('inf')
    dist = {s: 0}; dq = deque([s])
    while dq:
        u = dq.popleft()
        for v, w in adjw[u]:
            nd = dist[u] + w
            if nd < dist.get(v, INF):
                dist[v] = nd
                dq.appendleft(v) if w == 0 else dq.append(v)
    return dist


def floyd(n, edges):
    """暴力真值：全源最短路 O(n^3)。"""
    INF = float('inf')
    D = [[INF] * n for _ in range(n)]
    for i in range(n):
        D[i][i] = 0
    for u, v, w in edges:
        D[u][v] = min(D[u][v], w)
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if D[i][k] + D[k][j] < D[i][j]:
                    D[i][j] = D[i][k] + D[k][j]
    return D


rnd_g = random.Random(7)
for _ in range(300):
    n = rnd_g.randint(1, 8)
    edges = [(rnd_g.randrange(n), rnd_g.randrange(n), rnd_g.choice([0, 1]))
             for _ in range(rnd_g.randint(0, 14))]
    adj  = defaultdict(list); adjw = defaultdict(list)
    for u, v, w in edges:
        adj[u].append(v); adjw[u].append((v, w))
    for i in range(n):
        adj[i]; adjw[i]
    D = floyd(n, edges)
    for s in range(n):
        dd = dijkstra(adjw, s)
        d01 = bfs01(adjw, s)
        for t in range(n):
            truth = D[s][t]
            assert (dd.get(t, float('inf')) == truth), ('dijkstra', edges, s, t)
            assert (d01.get(t, float('inf')) == truth), ('bfs01', edges, s, t)
        # 边权全 1 时 BFS 必须等于 Dijkstra
        adj1 = defaultdict(list)
        for u, v, _ in edges:
            adj1[u].append(v)
        for i in range(n):
            adj1[i]
        b = bfs_dist(adj1, s)
        d1 = dijkstra({i: [(v, 1) for v in adj1[i]] for i in range(n)}, s)
        assert b == {k: int(v) for k, v in d1.items()}, ('bfs vs dijkstra', edges, s)
print('✅ 300 张随机图 x 全部源点：Dijkstra、0-1 BFS 与 Floyd 全源真值完全一致；')
print('   边权全 1 时 BFS == Dijkstra。')

# 「出队才标记」的代价：入队次数
star = defaultdict(list)
for i in range(1, 60):
    star[0].append(i)
    for j in range(1, 60):
        star[i].append(j)
_, pushes = bfs_dist_buggy(star, 0)
print('\n稠密图上「出队才标记」的入队次数 =', pushes, ' vs 正确版最多 V-1 =', 59)
assert pushes > 20 * 59
print('   ↑ 队列规模从 O(V) 涨到 O(E)。在网格上这是 4 倍常数，在稠密图上是数量级差异。')

In [ ]:
# ── 多源 BFS = 距离变换（CV 里天天用的那个）──

def distance_transform(mask):
    """多源 BFS：每个像素到最近前景像素（mask==1）的 4 邻域步数。
       所有源点**一次性入队** ⇒ 一遍 BFS 就得到「到最近源」的距离。"""
    H, W = mask.shape
    INF = 10**9
    dist = np.full((H, W), INF, dtype=np.int64)
    q = deque()
    for i in range(H):
        for j in range(W):
            if mask[i, j]:
                dist[i, j] = 0; q.append((i, j))
    while q:
        i, j = q.popleft()
        for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            ni, nj = i + di, j + dj
            if 0 <= ni < H and 0 <= nj < W and dist[ni, nj] > dist[i, j] + 1:
                dist[ni, nj] = dist[i, j] + 1
                q.append((ni, nj))
    return dist


def distance_transform_brute(mask):
    """暴力真值：无障碍网格上，4 邻域最短步数就是到最近源点的曼哈顿距离。"""
    H, W = mask.shape
    src = [(i, j) for i in range(H) for j in range(W) if mask[i, j]]
    out = np.full((H, W), 10**9, dtype=np.int64)
    for i in range(H):
        for j in range(W):
            if src:
                out[i, j] = min(abs(i - a) + abs(j - b) for a, b in src)
    return out


rnd_dt = random.Random(11)
for _ in range(200):
    H, W = rnd_dt.randint(1, 7), rnd_dt.randint(1, 7)
    m = (np.array([[rnd_dt.random() < 0.25 for _ in range(W)] for _ in range(H)])).astype(np.int64)
    assert np.array_equal(distance_transform(m), distance_transform_brute(m))
print('✅ 200 张随机 mask：多源 BFS 的距离场 == 到最近前景的曼哈顿距离（暴力真值）。')

demo = np.zeros((5, 9), dtype=np.int64); demo[2, 1] = 1; demo[0, 7] = 1
print('\nmask（1 = 前景）:')
print(demo)
print('距离变换:')
print(distance_transform(demo))
print('\n📌 一次 BFS 解决 k 个源点，不需要跑 k 次单源 —— 这就是多源 BFS 的全部价值。')
print('   CV 用途：边缘距离场、骨架化、watershed 的种子扩张、mask 空洞的距离判据。')

## 5 · 网格连通域标记：两条路线对拍

**「岛屿数量」= connected component labeling（CCL）。同一个算法，两个名字。**

- 路线 ①：**BFS/DFS 泛洪一遍法** —— 代码短、常数小；递归版在大连通域上会爆栈
- 路线 ②：**逐像素两遍扫描 + 并查集** —— 顺序访存、可分块并行、内存有确定上界，
  是工业界与 GPU 实现的主流（Hoshen–Kopelman 家族）

两条路线的标号必须**逐像素完全一致**（都按行优先的首次出现顺序编号），下面对拍验证。

In [ ]:
NBR4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
NBR8 = NBR4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))


def ccl_flood(mask, nbrs=NBR4):
    """路线①：BFS 泛洪。返回 (标号图, 连通域个数)；标号 0 = 背景，1..k 按行优先首次出现编号。"""
    H, W = mask.shape
    lab = np.zeros((H, W), dtype=np.int64)
    k = 0
    for i in range(H):
        for j in range(W):
            if mask[i, j] and lab[i, j] == 0:
                k += 1
                lab[i, j] = k; q = deque([(i, j)])      # 入队即标号
                while q:
                    y, x = q.popleft()
                    for dy, dx in nbrs:
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < H and 0 <= nx < W and mask[ny, nx] and lab[ny, nx] == 0:
                            lab[ny, nx] = k; q.append((ny, nx))
    return lab, k


def ccl_two_pass(mask, nbrs=NBR4):
    """路线②：逐像素两遍扫描 + 并查集。
       这里内联一个最小并查集（只做路径压缩）；完整版（按秩合并 + 复杂度）见第 6 节。"""
    H, W = mask.shape
    parent = [0]                                        # 标号从 1 开始，parent[0] 占位

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)           # 小标号当代表元 ⇒ 结果确定

    # 只看「已经扫过」的邻居：dy<0，或同一行的 dx<0
    prev_nbrs = [(dy, dx) for dy, dx in nbrs if dy < 0 or (dy == 0 and dx < 0)]
    tmp = np.zeros((H, W), dtype=np.int64)
    nxt = 0
    for i in range(H):                                  # 第一遍：给临时标号，冲突就 union
        for j in range(W):
            if not mask[i, j]:
                continue
            found = []
            for dy, dx in prev_nbrs:
                ni, nj = i + dy, j + dx
                if 0 <= ni < H and 0 <= nj < W and tmp[ni, nj]:
                    found.append(int(tmp[ni, nj]))
            if not found:
                nxt += 1; parent.append(nxt); tmp[i, j] = nxt
            else:
                lo = min(found)
                tmp[i, j] = lo
                for f in found:
                    union(lo, f)                        # 同一个域被给了两个标号 → 记下等价
    remap, k = {}, 0                                    # 第二遍：代表元 → 1..k
    lab = np.zeros((H, W), dtype=np.int64)
    for i in range(H):
        for j in range(W):
            if tmp[i, j]:
                r = find(int(tmp[i, j]))
                if r not in remap:
                    k += 1; remap[r] = k
                lab[i, j] = remap[r]
    return lab, k


def canon(lab):
    """把标号规范化成「行优先首次出现顺序」，让两种实现可以逐像素比较。"""
    remap, nxt = {}, 0
    out = np.zeros_like(lab)
    for i in range(lab.shape[0]):
        for j in range(lab.shape[1]):
            v = int(lab[i, j])
            if v:
                if v not in remap:
                    nxt += 1; remap[v] = nxt
                out[i, j] = remap[v]
    return out


rnd_c = random.Random(13)
for nbrs, name in ((NBR4, '4 邻域'), (NBR8, '8 邻域')):
    for _ in range(400):
        H, W = rnd_c.randint(1, 8), rnd_c.randint(1, 8)
        m = np.array([[1 if rnd_c.random() < 0.45 else 0 for _ in range(W)]
                      for _ in range(H)], dtype=np.int64)
        l1, k1 = ccl_flood(m, nbrs)
        l2, k2 = ccl_two_pass(m, nbrs)
        assert k1 == k2, (name, m, k1, k2)
        assert np.array_equal(canon(l1), canon(l2)), (name, m)
    print('  ✅ %s：400 张随机 mask，泛洪法与两遍扫描法**逐像素标号一致**' % name)

In [ ]:
# ── 4 邻域 vs 8 邻域：最小反例，以及它在 TSR 里的后果 ──
diag = np.array([[1, 0],
                 [0, 1]], dtype=np.int64)
print('mask =\n', diag)
print('4 邻域连通域个数 =', ccl_flood(diag, NBR4)[1], '（对角不相连）')
print('8 邻域连通域个数 =', ccl_flood(diag, NBR8)[1], '（对角相连）')
assert ccl_flood(diag, NBR4)[1] == 2 and ccl_flood(diag, NBR8)[1] == 1


def components_stats(lab, k):
    """连通域 → 面积 + 外接框（mask → bbox，分割结果转检测框的标准后处理）。"""
    out = []
    for c in range(1, k + 1):
        ys, xs = np.nonzero(lab == c)
        out.append({'label': c, 'area': int(len(ys)),
                    'bbox': (int(ys.min()), int(xs.min()), int(ys.max()), int(xs.max()))})
    return out


# 合成一张 mask：两个方块 + 一条 1 像素宽的斜线（模拟标志牌上的斜向反光条）
m = np.zeros((12, 20), dtype=np.int64)
m[1:5, 1:5] = 1                       # 方块 A
m[1:5, 7:11] = 1                      # 方块 B（与 A 之间隔一列背景）
for t in range(6):
    m[6 + t, 12 + t] = 1              # 斜线：只在对角相邻
print('\n合成 mask：')
print(m)
for nbrs, name in ((NBR4, '4 邻域'), (NBR8, '8 邻域')):
    lab, k = ccl_flood(m, nbrs)
    st = components_stats(lab, k)
    print('\n%s → %d 个连通域' % (name, k))
    for d in st:
        print('   label %d  area %3d  bbox(y0,x0,y1,x1) = %s' % (d['label'], d['area'], d['bbox']))
    assert sum(d['area'] for d in st) == int(m.sum())      # 面积之和 == 前景像素数
lab4, k4 = ccl_flood(m, NBR4)
lab8, k8 = ccl_flood(m, NBR8)
assert (k4, k8) == (8, 3), (k4, k8)
print('\n❗ 后果（真实会上线的 bug）：')
print('   · 4 邻域把那条斜向反光条切成 6 个 1 像素碎块 → 面积阈值一过滤就全丢了')
print('   · 8 邻域会把「仅在角上接触」的两个目标合成一个 → 输出一个巨大的错框')
print('   拓扑事实：前景用 8 邻域 ⇔ 背景必须用 4 邻域，否则前景连通与背景连通自相矛盾')
print('   （Rosenfeld 1970）。面试里说出这一条，比多做十道岛屿题有用。')

## 6 · 并查集：优化的必要性、以及 bbox 聚类

两个优化解决**两个不同的问题**：
**按秩合并**防止树链化（把树高压到 $O(\log n)$）；**路径压缩**把访问过的路径拍平（摊还近似常数）。
下面先把「不优化会怎样」跑成数字，再用它做 bbox 聚类。

In [ ]:
class DSU:
    """路径压缩 + 按秩合并。steps 统计 find 的爬升步数，用来验证复杂度。"""
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.size = [1] * n
        self.count = n                     # 集合个数：每次成功 union 减 1
        self.steps = 0

    def find(self, x):
        root = x
        while self.parent[root] != root:   # 先找到根
            root = self.parent[root]; self.steps += 1
        while self.parent[x] != root:      # 路径压缩：整条链直接挂到 root
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                   # 已同集合 —— 这个返回值可以直接用来检测环
        if self.rank[ra] < self.rank[rb]:   # 按秩合并：矮树挂到高树下，树高不增
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1             # 只有等高相并，树高才 +1
        self.count -= 1
        return True


class DSUNaive:
    """❌ 无优化版：不压缩、不按秩。用来把「为什么需要优化」跑成数字。"""
    def __init__(self, n):
        self.parent = list(range(n)); self.steps = 0
    def find(self, x):
        while self.parent[x] != x:
            x = self.parent[x]; self.steps += 1
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


n = 2000
naive, opt = DSUNaive(n), DSU(n)
for i in range(1, n):                      # 最坏输入：union(i, i-1) 让朴素版长成一条链
    naive.union(i, i - 1); opt.union(i, i - 1)
naive.steps = opt.steps = 0
for i in range(n):
    naive.find(i); opt.find(i)
print('n = %d，最坏 union 序列后做 n 次 find：' % n)
print('  无优化   总爬升 %9d 步  ≈ n^2/2 = %d' % (naive.steps, n * n // 2))
print('  优化后   总爬升 %9d 步' % opt.steps)
print('  差距 %.0f 倍' % (naive.steps / max(opt.steps, 1)))
assert naive.steps > 0.4 * n * n
assert opt.steps < 5 * n
assert naive.steps > 100 * max(opt.steps, 1)
assert opt.count == 1 and opt.size[opt.find(0)] == n
print('\n✅ 两个优化一起用，摊还代价是 O(alpha(n))，alpha(n) <= 4 对一切实际的 n。')
print('   面试标准答法：「摊还近似常数，严格是 O(alpha(n))；这个界是 Tarjan 证明的，而且是紧的。」')

# 与暴力（BFS 求连通分量）对拍
def components_brute(n, edges):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    lab, c = [-1] * n, 0
    for s in range(n):
        if lab[s] != -1:
            continue
        q = deque([s]); lab[s] = c
        while q:
            u = q.popleft()
            for v in adj[u]:
                if lab[v] == -1:
                    lab[v] = c; q.append(v)
        c += 1
    return c, lab

def canon_labels(lab):
    remap, nxt, out = {}, 0, []
    for v in lab:
        if v not in remap:
            remap[v] = nxt; nxt += 1
        out.append(remap[v])
    return out

rnd_u = random.Random(17)
for _ in range(1000):
    nn = rnd_u.randint(1, 12)
    edges = [(rnd_u.randrange(nn), rnd_u.randrange(nn)) for _ in range(rnd_u.randint(0, 15))]
    d = DSU(nn)
    for u, v in edges:
        d.union(u, v)
    cb, lb = components_brute(nn, edges)
    assert d.count == cb, (nn, edges)
    assert canon_labels([d.find(i) for i in range(nn)]) == canon_labels(lb)
print('✅ 1000 张随机图：并查集的连通分量个数与标号（规范化后）都与 BFS 暴力法一致。')

In [ ]:
# ══════════ 并查集的真实战场：bbox 聚类 ══════════
# 场景：切片推理（C57-04）后，同一个标志牌被多个切片各检出一次，需要**合并碎片**；
#       或多相机结果融合。注意这是「合并」而不是「筛选」——所以用并查集，不是 NMS。

def iou_matrix(boxes):
    """N x N 的 IoU 矩阵（向量化）。boxes = [[x1,y1,x2,y2], ...]"""
    b = np.asarray(boxes, dtype=np.float64)
    ix1 = np.maximum(b[:, None, 0], b[None, :, 0]); iy1 = np.maximum(b[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(b[:, None, 2], b[None, :, 2]); iy2 = np.minimum(b[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(area[:, None] + area[None, :] - inter, 1e-9)


def cluster_dsu(boxes, thr):
    """IoU > thr 就 union ⇒ 连通分量即簇。返回规范化标号。"""
    n = len(boxes)
    if n == 0:
        return []
    M = iou_matrix(boxes)
    d = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if M[i, j] > thr:
                d.union(i, j)
    return canon_labels([d.find(i) for i in range(n)])


def cluster_pairwise_brute(boxes, thr):
    """暴力对照：逐对比较建邻接表 + BFS 求连通分量（完全不用并查集）。"""
    n = len(boxes)
    if n == 0:
        return []
    M = iou_matrix(boxes)
    edges = [(i, j) for i in range(n) for j in range(i + 1, n) if M[i, j] > thr]
    _, lab = components_brute(n, edges)
    return canon_labels(lab)


def merge_cluster_boxes(boxes, labels):
    """每个簇取外接框（跨片合并的标准做法之一；也可以按分数加权平均）。"""
    b = np.asarray(boxes, dtype=np.float64)
    out = []
    for c in range(max(labels) + 1 if labels else 0):
        sel = b[[i for i, l in enumerate(labels) if l == c]]
        out.append([sel[:, 0].min(), sel[:, 1].min(), sel[:, 2].max(), sel[:, 3].max()])
    return out


rnd_bb = random.Random(19)
for _ in range(500):
    n = rnd_bb.randint(0, 10)
    bs = []
    for _ in range(n):
        x, y = rnd_bb.randint(0, 40), rnd_bb.randint(0, 40)
        w, h = rnd_bb.randint(5, 25), rnd_bb.randint(5, 25)
        bs.append([x, y, x + w, y + h])
    for thr in (0.1, 0.3, 0.5, 0.7):
        assert cluster_dsu(bs, thr) == cluster_pairwise_brute(bs, thr), (bs, thr)
print('✅ 500 组随机框 x 4 个阈值：并查集聚类与逐对比较 + BFS 暴力法结果完全一致。')

# 计时：建边是 O(N^2)，这是聚类法的真实瓶颈（不是并查集）
for N in (200, 400, 800):
    bs = [[i % 50 * 7, i // 50 * 7, i % 50 * 7 + 30, i // 50 * 7 + 30] for i in range(N)]
    t0 = time.perf_counter(); cluster_dsu(bs, 0.5); dt = time.perf_counter() - t0
    print('  N=%4d  聚类耗时 %6.1f ms  （O(N^2) 建边主导，并查集部分近似 O(N)）' % (N, dt * 1000))

In [ ]:
# ══════════ chaining：聚类法的失效模式（必须知道的那个坑）══════════
# 构造一排等间距的框：相邻 IoU = 0.6，隔一个 IoU = 0.333，隔三个 IoU = 0。
BOXES = np.array([[25.0 * i, 0.0, 25.0 * i + 100.0, 100.0] for i in range(6)])
SCORES = np.array([0.95, 0.94, 0.93, 0.92, 0.91, 0.90])
M = iou_matrix(BOXES)
print('IoU 矩阵（保留两位）:')
print(np.round(M, 3))
assert abs(M[0, 1] - 0.6) < 1e-9 and abs(M[0, 2] - 1 / 3) < 1e-9 and M[0, 4] == 0.0

thr = 0.5
labels = cluster_dsu(BOXES, thr)
merged = merge_cluster_boxes(BOXES, labels)
print('\n【并查集聚类，thr=%.1f】簇标号 = %s  → %d 个簇' % (thr, labels, max(labels) + 1))
print('   合并后的框 =', [[round(v, 1) for v in bb] for bb in merged])
assert labels == [0] * 6, '6 个框被链式并成了一个簇'
assert merged[0] == [0.0, 0.0, 225.0, 100.0]
print('   ❗ 单框宽 100，合并后宽 225 —— 一个横跨整排的巨大错框。')
print('   根因：IoU(A,B)=0.6、IoU(B,C)=0.6，但 IoU(A,C)=0.333；')
print('        **IoU 关系不传递，而连通分量求的是传递闭包。**')


def greedy_nms(scores, M, thr):
    """贪心 NMS：按分数降序（用 (-score, idx) 做全序 tie-break，见模块 02）扫描，能留就留。"""
    order = sorted(range(len(scores)), key=lambda i: (-float(scores[i]), i))
    keep = []
    for i in order:
        if all(M[i, j] <= thr for j in keep):
            keep.append(i)
    return sorted(keep)


keep = greedy_nms(SCORES, M, thr)
print('\n【贪心 NMS，thr=%.1f】保留下标 = %s → %d 个框' % (thr, keep, len(keep)))
assert keep == [0, 2, 4]
print('   NMS 是「挑代表并抑制邻居」（不传递），聚类是「求连通分量」（传递）。')
print('   稀疏场景两者几乎一样，密集场景差别巨大 —— 这就是选错的代价。')
print('\n缓解 chaining 的三条工程手段：')
print('   ① 提高连边阈值，并额外要求类别一致')
print('   ② 更严格的连边条件（IoU 且 中心距 < d 且 尺度比 in [0.7, 1.4]）')
print('   ③ **只在切片边界带内允许合并** —— 把传递性限制在物理上确实可能是同一目标的区域内')
print('      （这是 SAHI 式跨片合并的标准做法，见 C57-04）')

## 7 · 拓扑排序：Kahn 分层、三色环检测、DAG 关键路径

Kahn 的**分层副产物**在工程上比拓扑序本身更有用：
同层任务无依赖 ⇒ **可并行**；层数 - 1 = **关键路径长度** = 无限并行度下的最短完成时间
（这正是 C63-04 的延迟预算分解在做的事）。

In [ ]:
def kahn_layers(n, edges):
    """返回分层列表（每层升序）；**有环返回 None**。
       结点 v 所在层号 = 从任一源点到 v 的最长路径长度。"""
    adj = [[] for _ in range(n)]
    indeg = [0] * n
    for u, v in edges:
        adj[u].append(v); indeg[v] += 1
    cur = sorted(i for i in range(n) if indeg[i] == 0)
    layers, seen = [], 0
    while cur:
        layers.append(cur); seen += len(cur)
        nxt = []
        for u in cur:
            for v in adj[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    nxt.append(v)
        cur = sorted(nxt)
    return None if seen != n else layers      # 没输出完 ⇒ 剩下的结点在环里


def find_cycle_edge(n, edges):
    """三色标记：返回环上的一条边 (u, v)，无环返回 None。
       只有「指向灰色结点」才是环；指向黑色只是 DAG 里的重复汇聚。"""
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    color = [0] * n                            # 0 白（未访问）1 灰（在递归栈上）2 黑（已完成）
    box = []

    def dfs(u):
        color[u] = 1
        for v in adj[u]:
            if color[v] == 1:
                box.append((u, v)); return True    # ★ back edge
            if color[v] == 0 and dfs(v):
                return True
        color[u] = 2
        return False

    for s in range(n):
        if color[s] == 0 and dfs(s):
            return box[0]
    return None


def has_cycle_wrong(n, edges):
    """❌ 只用一个 visited 集合 —— 把「重复汇聚」误判成环。"""
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    vis = set()
    def dfs(u):
        vis.add(u)
        for v in adj[u]:
            if v in vis or dfs(v):
                return True
        return False
    return any(dfs(s) for s in range(n) if s not in vis)


def longest_path_edges(n, edges):
    """DAG 上按拓扑序做一遍 DP，求最长路（边数）= 关键路径长度。"""
    layers = kahn_layers(n, edges)
    if layers is None:
        return None
    order = [u for lv in layers for u in lv]
    pre = defaultdict(list)
    for u, v in edges:
        pre[v].append(u)
    dp = {}
    for u in order:                            # 拓扑序保证前驱都算完了
        dp[u] = max([dp[p] + 1 for p in pre[u]], default=0)
    return max(dp.values(), default=0)


# 菱形图：0→1, 0→2, 1→3, 2→3 —— 无环，但错版会说有环
DIAMOND = [(0, 1), (0, 2), (1, 3), (2, 3)]
print('菱形图 0→1, 0→2, 1→3, 2→3')
print('  Kahn 分层        =', kahn_layers(4, DIAMOND))
print('  三色找环         =', find_cycle_edge(4, DIAMOND))
print('  只用 visited 的错版说有环？', has_cycle_wrong(4, DIAMOND), '  ← 错')
assert kahn_layers(4, DIAMOND) == [[0], [1, 2], [3]]
assert find_cycle_edge(4, DIAMOND) is None
assert has_cycle_wrong(4, DIAMOND) is True

CYC = [(0, 1), (1, 2), (2, 0)]
print('\n三元环 0→1→2→0：Kahn =', kahn_layers(3, CYC), ' 环上的边 =', find_cycle_edge(3, CYC))
assert kahn_layers(3, CYC) is None and find_cycle_edge(3, CYC) is not None

# 与全排列暴力对拍：有拓扑序 ⟺ 无环
rnd_p = random.Random(23)
for _ in range(600):
    nn = rnd_p.randint(1, 6)
    edges = list({(rnd_p.randrange(nn), rnd_p.randrange(nn)) for _ in range(rnd_p.randint(0, 10))})
    edges = [(u, v) for u, v in edges if u != v]
    layers = kahn_layers(nn, edges)
    ok_brute = any(all(perm.index(u) < perm.index(v) for u, v in edges)
                   for perm in itertools.permutations(range(nn)))
    assert (layers is not None) == ok_brute, (nn, edges)
    assert (find_cycle_edge(nn, edges) is None) == ok_brute
    if layers is not None:
        flat = [u for lv in layers for u in lv]
        assert sorted(flat) == list(range(nn))
        pos = {u: i for i, u in enumerate(flat)}
        assert all(pos[u] < pos[v] for u, v in edges), '展平后必须是合法拓扑序'
        li = {u: i for i, lv in enumerate(layers) for u in lv}
        assert all(li[u] < li[v] for u, v in edges), '层号必须严格递增'
        assert longest_path_edges(nn, edges) == len(layers) - 1, '关键路径 = 层数 - 1'
print('\n✅ 600 张随机图：Kahn / 三色标记 / 全排列暴力三者对「是否有环」完全一致；')
print('   分层是合法拓扑序，且「关键路径边数 == 层数 - 1」处处成立。')

## 8 · 把 NMS 表述成图上的极大独立集

建模：顶点 = 候选框，$\mathrm{IoU}>\tau$ 连边 ⇒ **冲突图** $G$。于是

- 贪心 NMS 的输出是 $G$ 的一个**极大独立集**（maximal independent set）
- 我们真正想要的是**最大权独立集** MWIS（$\max\sum_{i\in S}s_i$，两两无边）—— 一般图上 **NP-hard**
- 按权降序的贪心是 MWIS 的标准近似，且有保证：$w(\text{greedy}) \ge \mathrm{OPT}/(\Delta+1)$，
  $\Delta$ = 冲突图最大度

下面把这四条**全部跑成断言**：验证独立性、验证极大性、暴力枚举求 OPT、量化次优、验证近似比。

In [ ]:
def is_independent(keep, M, thr):
    return all(M[i, j] <= thr for a, i in enumerate(keep) for j in keep[a + 1:])

def is_maximal(keep, M, thr, n):
    """极大：任何未被保留的框，加进来都会破坏独立性。"""
    return all(any(M[i, j] > thr for j in keep) for i in range(n) if i not in keep)

def mwis_brute(scores, M, thr):
    """暴力枚举所有子集求最大权独立集（n <= 16）。返回 (最优权重, 最优集合)。"""
    n = len(scores)
    best, arg = -1.0, None
    for mask in range(1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        if is_independent(idx, M, thr):
            w = float(sum(scores[i] for i in idx))
            if w > best:
                best, arg = w, idx
    return best, arg

def max_degree(M, thr, n):
    return max([sum(1 for j in range(n) if j != i and M[i, j] > thr) for i in range(n)], default=0)


# 先在上一节那排框上验证
keep = greedy_nms(SCORES, M, thr)
opt_w, opt_set = mwis_brute(SCORES, M, thr)
g_w = float(SCORES[keep].sum())
delta = max_degree(M, thr, len(SCORES))
print('那一排 6 个框（thr=0.5）：')
print('  贪心 NMS 保留 %s  权重 %.2f' % (keep, g_w))
print('  暴力 MWIS  保留 %s  权重 %.2f  最大度 Delta = %d' % (opt_set, opt_w, delta))
assert is_independent(keep, M, thr), '贪心输出必须是独立集'
assert is_maximal(keep, M, thr, len(SCORES)), '贪心输出必须是极大的'
assert g_w <= opt_w + 1e-12
assert g_w >= opt_w / (delta + 1) - 1e-12
print('  ✅ 独立性、极大性、g <= OPT、g >= OPT/(Delta+1) 全部成立')

# 随机搜索：贪心什么时候严格次优？
rnd_m = random.Random(29)
n_sub, worst, worst_case = 0, 1.0, None
for _ in range(3000):
    n = rnd_m.randint(2, 9)
    bs, sc = [], []
    for _ in range(n):
        x, y = rnd_m.randint(0, 30), rnd_m.randint(0, 30)
        w, h = rnd_m.randint(8, 24), rnd_m.randint(8, 24)
        bs.append([x, y, x + w, y + h]); sc.append(round(rnd_m.uniform(0.5, 1.0), 3))
    sc = np.array(sc); Mi = iou_matrix(bs); th = 0.4
    kp = greedy_nms(sc, Mi, th)
    ow, _ = mwis_brute(sc, Mi, th)
    gw = float(sc[kp].sum())
    d = max_degree(Mi, th, n)
    assert is_independent(kp, Mi, th) and is_maximal(kp, Mi, th, n)
    assert gw <= ow + 1e-9, '贪心不可能超过最优'
    assert gw >= ow / (d + 1) - 1e-9, '(Delta+1)-近似界必须成立'
    if gw < ow - 1e-9:
        n_sub += 1
        if gw / ow < worst:
            worst, worst_case = gw / ow, (bs, sc.tolist(), th, kp, ow, gw, d)
print('\n3000 组随机场景（thr=0.4）：')
print('  贪心严格次优的比例 = %d/3000 = %.1f%%' % (n_sub, 100 * n_sub / 3000))
print('  最差比值 greedy/OPT = %.4f' % worst)
assert n_sub > 0, '必须能找到贪心次优的例子'
assert worst < 1.0
bs_w, sc_w, th_w, kp_w, ow_w, gw_w, d_w = worst_case
print('  最差例子：%d 个框，最大度 Delta=%d，贪心 %.3f vs 最优 %.3f' % (len(bs_w), d_w, gw_w, ow_w))
print('\n📌 60 秒答法（背下来）：')
print('   「把每个框当顶点、IoU 超阈值连边，NMS 的输出就是这张冲突图上的一个极大独立集；')
print('    我们想要的是最大权独立集，但一般图上这是 NP-hard 的，所以贪心 NMS 是它的近似算法，')
print('    近似比与冲突图的最大度有关 —— 这解释了为什么密集场景下 NMS 更容易出问题。')
print('    如果冲突只来自一维区间重叠，图是区间图，可以用 DP 精确解，那就是加权区间调度。」')
print('   （一维精确解在模块 04；NMS 的手撕实现在 C61-05。）')

## ✏️ 练习 1：用并查集做 bbox 聚类

实现 `cluster_boxes(boxes, iou_thr)`：

- 输入 `boxes = [[x1, y1, x2, y2], ...]`（可能为空），`iou_thr` 是连边阈值
- **IoU > iou_thr 就把两个框并进同一簇**（严格大于，与本节的实现口径一致）
- 返回长度为 `N` 的标号列表，标号**按第一次出现的顺序规范化**为 `0, 1, 2, ...`
  （这样结果唯一确定，可以直接对拍）
- 必须用并查集；建边可以用 `iou_matrix(boxes)`

**先在心里回答**：这道题为什么不能用 NMS？（提示：合并 vs 筛选，以及 chaining）

In [ ]:
def cluster_boxes(boxes, iou_thr):
    # TODO:
    #   n = len(boxes); if n == 0: return []
    #   M = iou_matrix(boxes)
    #   d = DSU(n)
    #   对所有 i < j：M[i, j] > iou_thr 就 d.union(i, j)
    #   return canon_labels([d.find(i) for i in range(n)])
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert cluster_boxes([], 0.5) == []
assert cluster_boxes([[0, 0, 10, 10]], 0.5) == [0]
assert cluster_boxes([[0, 0, 10, 10], [100, 100, 110, 110]], 0.5) == [0, 1]

# 那一排链式相连的框：必须全部并成一簇（这是 chaining，也是这道题的重点）
assert cluster_boxes(BOXES.tolist(), 0.5) == [0] * 6
# 阈值提高到 0.65 之后，相邻 IoU=0.6 不再连边 ⇒ 六个独立簇
assert cluster_boxes(BOXES.tolist(), 0.65) == [0, 1, 2, 3, 4, 5]

_r1 = random.Random(101)
for _ in range(500):
    n = _r1.randint(0, 10)
    bs = []
    for _ in range(n):
        x, y = _r1.randint(0, 40), _r1.randint(0, 40)
        w, h = _r1.randint(5, 25), _r1.randint(5, 25)
        bs.append([x, y, x + w, y + h])
    for thr in (0.1, 0.3, 0.5, 0.7):
        lab = cluster_boxes(bs, thr)
        assert lab == cluster_pairwise_brute(bs, thr), (bs, thr)
        # 结构自检：同簇 ⟺ 在「IoU>thr」图上连通
        if n:
            Mi = iou_matrix(bs)
            for i in range(n):
                for j in range(n):
                    if Mi[i, j] > thr:
                        assert lab[i] == lab[j], '直接相连的框必须同簇'

_t0 = time.perf_counter()
_big = [[i % 60 * 6, i // 60 * 6, i % 60 * 6 + 28, i // 60 * 6 + 28] for i in range(300)]
_lab = cluster_boxes(_big, 0.5)
_dt = time.perf_counter() - _t0
assert _dt < 5.0, 'N=300 还要 5 秒以上 ⇒ 很可能对每对都跑了一次 BFS'
print('✅ 练习 1 通过：500 组随机框 x 4 个阈值与暴力法一致 + chaining 用例 + N=300 用时 %.0f ms'
      % (_dt * 1000))
print('   面试话术：「跨片合并要的是把碎片**合并**，所以用并查集求连通分量；')
print('   但 IoU 不传递，密集场景会 chaining —— 我会加中心距和尺度比约束，或只在边界带内合并。」')

## ✏️ 练习 2：迭代版网格连通域计数

实现 `count_components_iter(mask, conn=4)`：返回 `mask` 中值为 1 的连通域个数。

- `conn` 取 `4` 或 `8`，分别对应 4 邻域与 8 邻域
- **必须是迭代实现（显式栈或队列），不许递归** ——
  自测会在一张 $300\times300$ 的全 1 网格上跑，递归版必然 `RecursionError`
- 不要修改输入 `mask`（自测会检查）
- 空网格（0 行或 0 列）返回 0

In [ ]:
def count_components_iter(mask, conn=4):
    # TODO:
    #   nbrs = NBR4 if conn == 4 else NBR8
    #   H, W = mask.shape；seen = np.zeros_like(mask, dtype=bool)
    #   行优先扫描，遇到未访问的前景像素：计数 +1，然后用 **显式栈/队列** 泛洪
    #   泛洪时：入栈前判越界 + 判前景 + 判未访问，并**入栈时就置 seen=True**
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert count_components_iter(np.zeros((0, 0), dtype=np.int64)) == 0
assert count_components_iter(np.zeros((4, 4), dtype=np.int64)) == 0
assert count_components_iter(np.ones((4, 4), dtype=np.int64)) == 1
_diag = np.array([[1, 0], [0, 1]], dtype=np.int64)
assert count_components_iter(_diag, 4) == 2 and count_components_iter(_diag, 8) == 1

_r2 = random.Random(103)
for _ in range(400):
    H, W = _r2.randint(1, 8), _r2.randint(1, 8)
    m2 = np.array([[1 if _r2.random() < 0.45 else 0 for _ in range(W)]
                   for _ in range(H)], dtype=np.int64)
    before = m2.copy()
    for conn, nbrs in ((4, NBR4), (8, NBR8)):
        assert count_components_iter(m2, conn) == ccl_flood(m2, nbrs)[1], (m2, conn)
    assert np.array_equal(m2, before), '不许修改输入 mask'

# 大网格：单个连通域含 90000 个像素 ⇒ 递归版会 RecursionError（当前上限 %d）
_t0 = time.perf_counter()
_huge = np.ones((300, 300), dtype=np.int64)
assert count_components_iter(_huge, 4) == 1
_dt = time.perf_counter() - _t0
print('✅ 练习 2 通过：400 张随机 mask x 两种邻域与参考实现一致；')
print('   300x300 全 1 网格（90000 像素、递归深度会到 9e4）用时 %.0f ms 且未爆栈。' % (_dt * 1000))
print('   这就是「递归深度不可控时必须写迭代版」的验收标准。')

## ✏️ 练习 3：迭代求树的直径（stage 状态机）

实现 `tree_diameter_iter(root)`：返回树的直径（**两结点间最长路径的边数**），
**必须用显式栈迭代实现**，不许递归。

这是第 2 节 `subtree_sums_iter` 的直接变形：

- 栈里存 `(结点, stage)`，`stage ∈ {0, 1, 2}`
- 在 `stage == 2`（两个孩子都回来了）时：
  - `depth[node] = 1 + max(depth[left], depth[right])`（空孩子的深度算 0）
  - `best = max(best, depth[left] + depth[right])`（经过 node 的最长路径）
- 空树返回 0

自测会在一条深度约 2000 的链上跑 —— 递归版在那里必然爆栈。

In [ ]:
def tree_diameter_iter(root):
    # TODO：
    #   if root is None: return 0
    #   depth = {}; best = 0; st = [(root, 0)]
    #   while st:
    #       n, stage = st.pop()
    #       stage 0 -> 压回 (n,1)，再压左孩子 (left,0)
    #       stage 1 -> 压回 (n,2)，再压右孩子 (right,0)
    #       stage 2 -> dl = depth.get(id(n.left), 0); dr = depth.get(id(n.right), 0)
    #                  best = max(best, dl + dr); depth[id(n)] = 1 + max(dl, dr)
    #   return best
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert tree_diameter_iter(None) == 0
assert tree_diameter_iter(build([1])) == 0
assert tree_diameter_iter(build([1, 2, 3])) == 2
assert tree_diameter_iter(T) == diameter(T)

_r3 = random.Random(107)
for _ in range(1500):
    t3 = rand_tree(_r3, _r3.randint(0, 16))
    assert tree_diameter_iter(t3) == diameter(t3) == diameter_brute(t3), '与递归版和 BFS 暴力真值都要一致'

# 深链：递归版必爆栈，迭代版必须给出 N-1
_N = 2 * sys.getrecursionlimit()
_chain = None
for _v in range(_N):
    _chain = Node(_v, left=_chain)
_rec_ok = True
try:
    diameter(_chain)
except RecursionError:
    _rec_ok = False
_d = tree_diameter_iter(_chain)
unlink_chain(_chain); _chain = None
assert _rec_ok is False, '深度 %d 的链上递归版应当爆栈' % _N
assert _d == _N - 1, ('链的直径 = 结点数 - 1', _d, _N - 1)
print('✅ 练习 3 通过：1500 棵随机树与递归版/暴力真值三方一致；')
print('   深度 %d 的链上递归版爆栈，迭代版给出直径 %d。' % (_N, _d))
print('   记住这个骨架：它能把**任何**后序型递归（树形 DP、记忆化搜索）改成迭代。')

## ✏️ 练习 4：把 NMS 写成贪心极大独立集

实现 `greedy_mis(weights, edges)`：

- `weights[i]` 是顶点 `i` 的权重，`edges` 是**无向冲突边**列表 `[(u, v), ...]`
  （可能含重复边与自环，自环请忽略）
- 按**权重降序**扫描（权重并列时按**下标升序**，保证结果确定），
  与已选集合无冲突就选入
- 返回**升序**的下标列表

这就是 NMS 换一个接口写出来的样子：`weights` = 分数，`edges` = IoU 超阈值的框对。
自测会验证四条性质：**独立性、极大性、$w\le\mathrm{OPT}$、$w\ge\mathrm{OPT}/(\Delta+1)$**。

In [ ]:
def greedy_mis(weights, edges):
    # TODO:
    #   conflict = defaultdict(set)；把 edges 里 u != v 的双向加入
    #   order = sorted(range(len(weights)), key=lambda i: (-weights[i], i))
    #   keep = []；逐个考察 i：若 conflict[i] 与 keep 无交集则 keep.append(i)
    #   return sorted(keep)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
def _mwis_edges_brute(weights, edges):
    """暴力枚举全部子集求最大权独立集（n <= 14）。"""
    n = len(weights)
    E = {(min(u, v), max(u, v)) for u, v in edges if u != v}
    best, arg = -1.0, None
    for mask in range(1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        if all((min(i, j), max(i, j)) not in E
               for a, i in enumerate(idx) for j in idx[a + 1:]):
            w = float(sum(weights[i] for i in idx))
            if w > best:
                best, arg = w, idx
    return best, arg

def _deg(n, edges):
    d = defaultdict(set)
    for u, v in edges:
        if u != v:
            d[u].add(v); d[v].add(u)
    return d, max([len(d[i]) for i in range(n)], default=0)

assert greedy_mis([], []) == []
assert greedy_mis([1.0, 2.0, 3.0], []) == [0, 1, 2], '无边 ⇒ 全选'
assert greedy_mis([1.0], [(0, 0)]) == [0], '自环必须被忽略'
# 星形反例：贪心先拿中心（3），把三个叶子（各 2）全挡掉 ⇒ 3 < 6
_star_w, _star_e = [3.0, 2.0, 2.0, 2.0], [(0, 1), (0, 2), (0, 3)]
assert greedy_mis(_star_w, _star_e) == [0]
assert _mwis_edges_brute(_star_w, _star_e)[0] == 6.0
print('星形反例：贪心权重 3.0，最优 6.0 —— 贪心可以差到 2 倍。')

_r4 = random.Random(109)
_sub = 0
for _ in range(2000):
    n = _r4.randint(1, 10)
    w = [round(_r4.uniform(0.5, 1.0), 3) for _ in range(n)]
    es = [(_r4.randrange(n), _r4.randrange(n)) for _ in range(_r4.randint(0, 14))]
    keep4 = greedy_mis(w, es)
    d, delta = _deg(n, es)
    assert keep4 == sorted(set(keep4)), '返回值必须是升序且不重复'
    assert all(v not in d[u] for a, u in enumerate(keep4) for v in keep4[a + 1:]), '① 必须是独立集'
    assert all(any(j in d[i] for j in keep4) for i in range(n) if i not in keep4), '② 必须是极大的'
    ow, _ = _mwis_edges_brute(w, es)
    gw = sum(w[i] for i in keep4)
    assert gw <= ow + 1e-9, '③ 贪心不可能超过最优'
    assert gw >= ow / (delta + 1) - 1e-9, '④ (Delta+1)-近似界'
    if gw < ow - 1e-9:
        _sub += 1
print('✅ 练习 4 通过：2000 组随机冲突图，四条性质全部成立；')
print('   其中 %d 组（%.1f%%）贪心严格次优 —— 这就是「NMS 是 NP-hard 问题的贪心近似」的实证。'
      % (_sub, 100 * _sub / 2000))
assert _sub > 0

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def cluster_boxes(boxes, iou_thr):
    n = len(boxes)
    if n == 0:
        return []
    M = iou_matrix(boxes)              # O(N^2) 建边 —— 这才是真正的瓶颈
    d = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if M[i, j] > iou_thr:
                d.union(i, j)          # 并查集部分近似 O(N alpha(N))
    return canon_labels([d.find(i) for i in range(n)])
# 为什么不能用 NMS：NMS 是「挑代表并抑制邻居」（筛选），聚类是「求连通分量」（合并）。
# 代价：IoU 不传递 ⇒ 密集场景会 chaining，要靠中心距/尺度比/边界带来约束传递性。

In [ ]:
# 练习 2 参考答案
def count_components_iter(mask, conn=4):
    nbrs = NBR4 if conn == 4 else NBR8
    if mask.size == 0:
        return 0
    H, W = mask.shape
    seen = np.zeros((H, W), dtype=bool)
    cnt = 0
    for i in range(H):
        for j in range(W):
            if not mask[i, j] or seen[i, j]:
                continue
            cnt += 1
            seen[i, j] = True
            st = [(i, j)]              # 显式栈：深度多大都不会爆
            while st:
                y, x = st.pop()
                for dy, dx in nbrs:
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and mask[ny, nx] and not seen[ny, nx]:
                        seen[ny, nx] = True    # ★ 入栈时就标记，避免重复入栈
                        st.append((ny, nx))
    return cnt

In [ ]:
# 练习 3 参考答案
def tree_diameter_iter(root):
    if root is None:
        return 0
    depth = {}                         # id(结点) -> 向下最大深度（空 = 0）
    best = 0
    st = [(root, 0)]
    while st:
        n, stage = st.pop()
        if stage == 0:
            st.append((n, 1))
            if n.left:  st.append((n.left, 0))
        elif stage == 1:
            st.append((n, 2))
            if n.right: st.append((n.right, 0))
        else:                          # 两个孩子都回来了 —— 这就是"后序时刻"
            dl = depth.get(id(n.left), 0)
            dr = depth.get(id(n.right), 0)
            best = max(best, dl + dr)  # 答案：经过 n 的最长路径（边数）
            depth[id(n)] = 1 + max(dl, dr)   # 返回值：深度。两者不是一回事
    return best

In [ ]:
# 练习 4 参考答案
def greedy_mis(weights, edges):
    conflict = defaultdict(set)
    for u, v in edges:
        if u != v:                     # 自环忽略：一个框和自己不冲突
            conflict[u].add(v); conflict[v].add(u)
    # 全序键 (-权重, 下标)：并列时也有确定顺序（见模块 02 的确定性讨论）
    order = sorted(range(len(weights)), key=lambda i: (-weights[i], i))
    keep = []
    for i in order:
        if not conflict[i].intersection(keep):   # 与已选集合无冲突 ⇒ 选入
            keep.append(i)
    return sorted(keep)
# 性质：① 独立（构造保证）② 极大（被跳过的一定与某个已选冲突）
#      ③ w <= OPT ④ w >= OPT/(Delta+1)：每个被选中的点最多挡掉 Delta 个权重不超过它的点

---
## 🧪 真实工程胶囊：树 / 图 / 搜索速查卡

In [ ]:
RECIPE = r'''
# ======================================================================
# 面试与生产两用速查卡 · 递归转迭代 / 并查集 / 网格搜索 / NMS 图视角   （C62 模块 03）
# ======================================================================

# ---------- 1. 递归三要素（写代码前口述一遍） ----------
#   终止：最小子问题是什么、答案是什么           -> 漏了就 RecursionError
#   拆解：怎么变成"严格更小"的同类问题            -> 图上必须有 visited，否则没有递减度量
#   合并：子答案怎么拼成当前答案                  -> 先写一行契约注释：输入/返回/副作用
#   陷阱：**返回值 != 答案**（直径题返回深度，答案挂在外部变量上）
#   陷阱：每层重算高度 = 重复计算，最坏 O(n^2)；正解是后序一趟把高度带上来

# ---------- 2. 递归转迭代：万能配方 ----------
#   栈元素 = (状态, 阶段)；阶段 k = "第 k 个递归调用点的返回位置"
#   stage 0 -> 压回 (n,1) 再压第一个子问题；stage 1 -> 压回 (n,2) 再压第二个；stage 2 -> 合并
#   子结果放在以"结点身份"为键的表里（树用 id(node)，图用结点编号）
def postorder_iter(root):
    out, st = [], ([(root, 0)] if root else [])
    while st:
        n, stage = st.pop()
        if   stage == 0: st.append((n, 1));  st.append((n.left, 0))  if n.left  else None
        elif stage == 1: st.append((n, 2));  st.append((n.right, 0)) if n.right else None
        else:            out.append(n.val)
    return out
# 什么时候**必须**转：① 深度可能到 1e4+  ② 需要可中断（帧预算耗尽就存栈下帧继续）
#                    ③ 需要可序列化/可限长内存
# 反面教材：sys.setrecursionlimit(10**6) —— 爆的是 C 栈，段错误、无异常、无日志

# ---------- 3. BFS / Dijkstra / 0-1 BFS ----------
# BFS：dist 字典兼作 visited；**入队时标记**（出队才标记会让队列从 O(V) 涨到 O(E)）
# 多源 BFS：所有源点一次性入队 = 距离变换（CV 里的 distance transform）
# Dijkstra：堆 + 惰性删除（不 decrease-key；出堆时 `if d > dist[u]: continue`）；边权须非负
# 0-1 BFS：deque，权 0 走 appendleft、权 1 走 append -> O(V+E)
# 选择信号：最短步数->BFS；所有路径/回溯->DFS；有代价->Dijkstra；有负权->Bellman-Ford

# ---------- 4. 并查集（默写级） ----------
class DSU:
    def __init__(self, n):
        self.p = list(range(n)); self.r = [0]*n; self.sz = [1]*n; self.count = n
    def find(self, x):
        root = x
        while self.p[root] != root: root = self.p[root]
        while self.p[x] != root: self.p[x], x = root, self.p[x]   # 路径压缩
        return root
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return False                                  # 返回值可直接用来检测环
        if self.r[ra] < self.r[rb]: ra, rb = rb, ra                # 按秩合并
        self.p[rb] = ra; self.sz[ra] += self.sz[rb]
        if self.r[ra] == self.r[rb]: self.r[ra] += 1
        self.count -= 1; return True
# 复杂度 O(m alpha(n))，alpha <= 4；两个优化各管一件事：按秩防链化、压缩降摊还
# 做不到：删边、分裂、查距离 -> 删边问题用**离线倒序处理**变成加边

# ---------- 5. 网格搜索 = 连通域标记（CCL） ----------
# 路线① BFS/DFS 泛洪：短、常数小；递归版在 1e6 像素的域上必爆栈
# 路线② 两遍扫描 + 并查集：顺序访存、可分块并行、内存有上界 -> 工业界与 GPU 主流
# 4 vs 8 邻域是**语义选择**：8 邻域会把角接触的两个目标并成一个巨框；
#                          4 邻域会把 1 像素宽的斜向结构切成碎块
# 拓扑事实：前景 8 邻域 <=> 背景必须 4 邻域（Rosenfeld 1970）
# 产物：面积（过滤噪点）、外接框（mask -> bbox）、距离场（多源 BFS）

# ---------- 6. 框聚类 vs NMS：合并还是筛选 ----------
# 并查集聚类 = 连通分量 = **传递闭包**  -> 用于跨片合并、多相机融合（要"拼碎片"）
# 贪心 NMS   = 极大独立集             -> 用于去重（要"挑代表"）
# chaining 失效：IoU(A,B)=.6, IoU(B,C)=.6, IoU(A,C)=0 却被并成一簇 -> 一排限速牌变一个巨框
# 三条缓解：提高阈值+同类别 / 加中心距与尺度比约束 / **只在切片边界带内允许合并**

# ---------- 7. NMS 的图论表述（60 秒讲稿） ----------
# 顶点=框，IoU>tau 连边 -> 冲突图 G
#   贪心 NMS 的输出 = G 的一个**极大独立集**
#   想要的是**最大权独立集 MWIS** -> 一般图 NP-hard -> 所以 NMS 是贪心近似
#   保证：w(greedy) >= OPT/(Delta+1)，Delta = 最大度
#         => Delta 小(稀疏)时几乎最优；Delta 大(密集)时才明显次优 —— 解释了密集场景为何变差
#   一维特例：冲突来自区间重叠 -> 区间图 -> DP 可**精确**解（加权区间调度，见模块 04）
#   绕开路线：一对一匹配的无 NMS 检测器（DETR / YOLOv10）—— 把"推理期去重"变成"训练期不产生重复"

# ---------- 8. 交卷前 30 秒自检 ----------
#   [ ] 递归：说清返回值语义；估一下最坏深度；深度不可控就改迭代
#   [ ] BFS：入队时标记；层数用 len(queue) 切层
#   [ ] 网格：越界判断写在入队前；4/8 邻域先问；是否允许改输入先问
#   [ ] 并查集：find 写迭代版；说出"摊还 O(alpha(n))"而不是"O(1)"
#   [ ] 环检测：三色标记，只有指向**灰色**才是环（单 visited 集合判不出来）
#   [ ] 边界：空树/空图/单结点/自环/重边/非连通图
'''
print(RECIPE)
for _tok in ['postorder_iter', 'class DSU', '入队时标记', 'Rosenfeld 1970',
             '极大独立集', 'OPT/(Delta+1)', '交卷前 30 秒自检']:
    assert _tok in RECIPE, _tok
print('\n（第 2、4 节可直接复制进项目；第 7 节建议在面试前一天读出声一遍。）')

### 小结

1. **递归 = 隐式栈；把栈显式化，控制流就变成了数据。**
   转换配方只有一条：栈里存 `(状态, 阶段)`，阶段编号就是「返回点」。
   本 notebook 用它把后序遍历、子树和、树的直径都改成了迭代版，
   并在深度 2000 的链上验证了「递归爆栈、迭代照常」。
   **必须转的三个信号是：深度不可控、需要可中断、需要可序列化** ——
   后两条在车端工程里比第一条更常见，却几乎不出现在算法教材里。

2. **网格题就是连通域标记，图题在 CV 里几乎总是换了个名字。**
   岛屿数量 = CCL，flood fill = 区域填充，多源 BFS = 距离变换，
   连通域外接框 = mask → bbox。而 **4/8 邻域是语义选择不是实现细节**：
   8 邻域会把角接触的两个标志牌并成一个巨框，4 邻域会把 1 像素宽的斜向反光条切成碎块。

3. **并查集要能默写，也要能说出它做不到什么。**
   两个优化各管一件事（按秩防链化、路径压缩降摊还），
   本 notebook 把「不优化」的代价跑成了 1000 倍的步数差。
   它不能删边、不能分裂、不能查距离——**删边问题的标准套路是离线倒序处理**。

4. **聚类与 NMS 是两件事：合并 vs 筛选。**
   并查集聚类求的是传递闭包，而 IoU 不传递，所以密集场景会 **chaining**——
   本 notebook 用一排框把「6 个宽 100 的框被并成一个宽 225 的巨框」跑了出来。
   跨片合并要用聚类（拼碎片），去重要用 NMS（挑代表），用错的症状非常典型。

5. **把 NMS 讲成「冲突图上的极大独立集」，是这一模块最值钱的 60 秒。**
   它一次性解释了：为什么 NMS 是贪心的、为什么排序不可省、
   为什么密集场景下它更容易出问题（近似比与最大度 $\Delta$ 有关）、
   以及为什么一维情形可以用 DP 精确解（模块 04）、
   为什么 DETR / YOLOv10 要用一对一匹配从根上绕开它（C53/C54）。
   **能把一个天天用的模块讲出结构，比多刷十道图论题有价值。**